# Phase 2 — Velocity & Geo-Risk Agents

**Developer 2 | Detection workstream**

Execution notebook: fit `VelocityAgent` and `GeoRiskAgent` on canonical transactions, inspect scores, and save outputs.

| Step | Cell(s) |
|---|---|
| 1 | Environment setup |
| 2 | Load canonical data |
| 3 | Fit & run Velocity agent |
| 4 | Fit & run Geo-Risk agent |
| 5 | Combined alert summary |
| 6 | Save outputs |

---
## 1. Environment Setup

In [ ]:
%%time
import importlib
import logging
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np

# --- Path bootstrap (local dev and Kaggle) ---
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('/kaggle/working/neural-sentinel')
assert (PROJECT_ROOT / 'src').exists(), (
    f'src/ not found under PROJECT_ROOT={PROJECT_ROOT}. '
    'Mount the Kaggle dataset or set PROJECT_ROOT manually.'
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Install only packages absent from this environment ---
_REQUIRED = {
    'sklearn':  'scikit-learn==1.6.1',
    'catboost': 'catboost==1.2.10',
    'pyarrow':  'pyarrow==20.0.0',
}
for _mod, _spec in _REQUIRED.items():
    if importlib.util.find_spec(_mod) is None:
        print(f'Installing {_spec} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _spec])

# --- Reproducibility ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Logging ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('phase2_notebook')

# --- GPU / accelerator detection ---
DEVICE = 'cpu'
try:
    import torch as _torch
    if _torch.cuda.is_available():
        DEVICE = 'cuda'
        logger.info('CUDA GPU: %s', _torch.cuda.get_device_name(0))
    else:
        logger.info('No GPU detected — CPU only (VelocityAgent and GeoRiskAgent do not require GPU)')
except ImportError:
    logger.info('PyTorch absent — CPU only')

if os.environ.get('TPU_NAME'):
    DEVICE = 'tpu'
    logger.info('Kaggle TPU: %s', os.environ['TPU_NAME'])

logger.info('PROJECT_ROOT: %s | device: %s', PROJECT_ROOT, DEVICE)

---
## 2. Load Canonical Data

Reads `transactions.parquet` from `data/interim/` (canonical schema).  
Falls back to the raw CSV in `data/original/original_data/` when the interim file is not yet produced by the data-pipeline workstream — this lets Phase 2 run independently during development.

In [ ]:
%%time
import pandas as pd
from src.utils.config import get_config

cfg = get_config()

INTERIM_PATH  = cfg.data_interim_dir / 'transactions.parquet'
FALLBACK_CSV  = cfg.data_original_dir / 'original_data' / 'transactions.csv'
KAGGLE_INPUT  = Path('/kaggle/input')

def _load_transactions() -> pd.DataFrame:
    """Load canonical transactions, trying interim Parquet then fallback CSV."""
    if INTERIM_PATH.exists():
        logger.info('Loading interim Parquet: %s', INTERIM_PATH)
        return pd.read_parquet(INTERIM_PATH)
    # Search Kaggle input directory for a transactions parquet
    if KAGGLE_INPUT.exists():
        matches = list(KAGGLE_INPUT.rglob('transactions.parquet'))
        if matches:
            logger.info('Kaggle input found: %s', matches[0])
            return pd.read_parquet(matches[0])
    if FALLBACK_CSV.exists():
        logger.warning(
            'Interim Parquet not found — loading raw CSV from %s. '
            'Run phase2_cleaning.ipynb first for full canonical features.',
            FALLBACK_CSV,
        )
        return pd.read_csv(FALLBACK_CSV, nrows=50_000)
    raise FileNotFoundError(
        f'No transactions data found. Expected {INTERIM_PATH} or {FALLBACK_CSV}.'
    )

transactions = _load_transactions()
logger.info('Loaded %d transactions, %d columns', len(transactions), transactions.shape[1])
print(f'Shape: {transactions.shape}')
print(f'Columns: {list(transactions.columns)}')
transactions.head(3)

In [ ]:
# Quick sanity-check: required columns and fraud rate
_REQUIRED_COLS = {'transaction_id'}
_missing = _REQUIRED_COLS - set(transactions.columns)
assert not _missing, f'Missing required columns: {_missing}'

if 'is_fraud' in transactions.columns:
    fraud_rate = transactions['is_fraud'].mean()
    print(f'Fraud rate: {fraud_rate:.4%} ({transactions["is_fraud"].sum():,} / {len(transactions):,})')
else:
    print('is_fraud column absent — evaluation metrics will be skipped')

print(f'Null counts (top 5):')
print(transactions.isnull().sum().sort_values(ascending=False).head())

---
## 3. Velocity Agent

`VelocityAgent` builds rolling account-level features (1 h / 6 h / 24 h / 7 d transaction count and sum, inter-arrival time, z-score) then fits an Isolation Forest to score unusual activity patterns. See `src/agents/velocity_agent.py` for full implementation.

In [ ]:
%%time
from src.agents.velocity_agent import VelocityAgent

velocity_agent = VelocityAgent(config=cfg, logger=logging.getLogger('velocity'))
velocity_agent.fit(transactions)
logger.info('VelocityAgent fitted — is_fitted=%s', velocity_agent.is_fitted)

In [ ]:
%%time
velocity_scores = velocity_agent.predict(transactions)
print(f'Output shape: {velocity_scores.shape}')
print(f'Columns: {list(velocity_scores.columns)}')
velocity_scores.head(5)

In [ ]:
# Score distribution and alert summary
v_alerts = velocity_scores['alert_flag'].sum()
v_alert_rate = velocity_scores['alert_flag'].mean()
print(f'Velocity alerts:  {v_alerts:,}  ({v_alert_rate:.3%} of transactions)')
print(f'Risk score stats:')
print(velocity_scores['risk_score'].describe().round(4))
print(f'\nReason code distribution:')
print(velocity_scores['reason_code'].value_counts())

In [ ]:
# Evaluation against ground truth (only when is_fraud is present)
if 'is_fraud' in transactions.columns:
    from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

    y_true = transactions['is_fraud'].astype(int).values
    y_score = velocity_scores['risk_score'].values
    y_pred  = velocity_scores['alert_flag'].values

    auc_roc = roc_auc_score(y_true, y_score)
    auc_pr  = average_precision_score(y_true, y_score)
    print(f'Velocity Agent — AUC-ROC: {auc_roc:.4f}  |  AUC-PR: {auc_pr:.4f}')
    print(classification_report(y_true, y_pred, target_names=['legit', 'suspicious'], zero_division=0))
else:
    print('Skipping evaluation: is_fraud not available')

In [ ]:
# Sample explain() calls on the top-scored transactions
top_velocity = velocity_scores.nlargest(3, 'risk_score')['transaction_id'].tolist()
print('Top velocity alerts:')
for tid in top_velocity:
    print(f'  {velocity_agent.explain(tid)}')

---
## 4. Geo-Risk Agent

`GeoRiskAgent` scores geographic and corridor risk using CatBoost when `is_fraud` labels are present, or falls back to a deterministic score built from corridor risk tiers, VPN flags, cross-border indicators, and IP-country mismatches. See `src/agents/geo_risk_agent.py` for full implementation.

In [ ]:
%%time
from src.agents.geo_risk_agent import GeoRiskAgent

geo_agent = GeoRiskAgent(config=cfg, logger=logging.getLogger('geo_risk'))
geo_agent.fit(transactions)
logger.info(
    'GeoRiskAgent fitted — use_catboost=%s  is_fitted=%s',
    geo_agent._use_catboost,
    geo_agent.is_fitted,
)

In [ ]:
%%time
geo_scores = geo_agent.predict(transactions)
print(f'Output shape: {geo_scores.shape}')
geo_scores.head(5)

In [ ]:
# Score distribution and alert summary
g_alerts = geo_scores['alert_flag'].sum()
g_alert_rate = geo_scores['alert_flag'].mean()
print(f'Geo-Risk alerts:  {g_alerts:,}  ({g_alert_rate:.3%} of transactions)')
print(f'Risk score stats:')
print(geo_scores['risk_score'].describe().round(4))
print(f'\nReason code distribution:')
print(geo_scores['reason_code'].value_counts())

In [ ]:
# Evaluation against ground truth
if 'is_fraud' in transactions.columns:
    y_true  = transactions['is_fraud'].astype(int).values
    y_score = geo_scores['risk_score'].values
    y_pred  = geo_scores['alert_flag'].values

    auc_roc = roc_auc_score(y_true, y_score)
    auc_pr  = average_precision_score(y_true, y_score)
    print(f'Geo-Risk Agent  — AUC-ROC: {auc_roc:.4f}  |  AUC-PR: {auc_pr:.4f}')
    print(classification_report(y_true, y_pred, target_names=['legit', 'suspicious'], zero_division=0))
else:
    print('Skipping evaluation: is_fraud not available')

In [ ]:
# Sample explain() calls on the top-scored transactions
top_geo = geo_scores.nlargest(3, 'risk_score')['transaction_id'].tolist()
print('Top geo-risk alerts:')
for tid in top_geo:
    print(f'  {geo_agent.explain(tid)}')

---
## 5. Combined Alert Summary

Joins both agent outputs on `transaction_id` to preview the input the Meta-Learner (Phase 4) will receive.

In [ ]:
combined = (
    velocity_scores[['transaction_id', 'risk_score', 'alert_flag', 'reason_code']]
    .rename(columns={'risk_score': 'velocity_score', 'alert_flag': 'velocity_flag', 'reason_code': 'velocity_reason'})
    .merge(
        geo_scores[['transaction_id', 'risk_score', 'alert_flag', 'reason_code']]
        .rename(columns={'risk_score': 'geo_score', 'alert_flag': 'geo_flag', 'reason_code': 'geo_reason'}),
        on='transaction_id',
        how='inner',
    )
)

if 'is_fraud' in transactions.columns:
    combined = combined.merge(
        transactions[['transaction_id', 'is_fraud']],
        on='transaction_id',
        how='left',
    )

print(f'Combined shape: {combined.shape}')
combined.head(5)

In [ ]:
# Transactions flagged by both agents simultaneously
both_flagged = combined.query('velocity_flag == 1 and geo_flag == 1')
print(f'Flagged by both agents: {len(both_flagged):,}  ({len(both_flagged)/len(combined):.3%})')

if 'is_fraud' in combined.columns:
    true_positive_overlap = both_flagged['is_fraud'].sum()
    print(f'Of those, confirmed suspicious: {true_positive_overlap:,}  ({true_positive_overlap/max(len(both_flagged),1):.3%})')

# Per-agent alert counts side by side
summary = pd.DataFrame({
    'agent':       ['velocity', 'geo_risk', 'both'],
    'alerts':      [
        combined['velocity_flag'].sum(),
        combined['geo_flag'].sum(),
        len(both_flagged),
    ],
    'alert_rate':  [
        combined['velocity_flag'].mean(),
        combined['geo_flag'].mean(),
        len(both_flagged) / len(combined),
    ],
})
print('\nAlert summary:')
print(summary.to_string(index=False))

---
## 6. Save Outputs

Writes per-agent alert scores to `data/interim/` as Parquet files for downstream phases.

In [ ]:
%%time
from tqdm.auto import tqdm

OUTPUT_DIR = cfg.data_interim_dir

# On Kaggle, write to working/ so outputs persist
if Path('/kaggle/working').exists() and not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path('/kaggle/working')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_outputs = {
    'velocity_scores.parquet':  velocity_scores,
    'geo_scores.parquet':       geo_scores,
    'combined_scores_p2.parquet': combined,
}

for filename, df in tqdm(_outputs.items(), desc='Saving'):
    path = OUTPUT_DIR / filename
    df.to_parquet(path, index=False)
    logger.info('Saved %s (%d rows) → %s', filename, len(df), path)

print('All outputs saved.')

In [ ]:
# Phase 2 complete
logger.info('Phase 2 complete — VelocityAgent and GeoRiskAgent fitted, scored, and saved.')
print('Phase 2 done. Outputs available in', OUTPUT_DIR)